# Ascend Phase 5 — DPO Preference Tuning

Fine-tunes your already-fine-tuned model on real user preferences.
Chosen = quests users completed. Rejected = quests users skipped.

**Prerequisites:**
- Phase 4 deployed for 2+ weeks
- ~500+ preference pairs collected
- `preferences.jsonl` uploaded to Drive

**Output:** New model `quest-model-merged-v2` on HuggingFace Hub

In [ ]:
# CELL 1: Install
!pip install -q transformers==4.41.0 peft==0.11.1 trl==0.9.4
!pip install -q bitsandbytes==0.43.1 datasets==2.19.0 accelerate==0.30.1
!pip install -q huggingface_hub
print('✅ Done')

In [ ]:
# CELL 2: Mount Drive + Load Preference Data
from google.colab import drive
drive.mount('/content/drive')

PREF_PATH = '/content/drive/MyDrive/ascend_ml/data/preferences.jsonl'

import json
from datasets import Dataset

with open(PREF_PATH) as f:
    pairs = [json.loads(l) for l in f]

print(f'Total preference pairs: {len(pairs):,}')

# DPO requires dataset with: prompt, chosen, rejected columns
dataset = Dataset.from_list(pairs)
split = dataset.train_test_split(test_size=0.05, seed=42)
train_ds = split['train']
eval_ds  = split['test']
print(f'Train: {len(train_ds)} | Eval: {len(eval_ds)}')

In [ ]:
# CELL 3: Load v1 model (your SFT model from Phase 2)
# DPO continues from where SFT left off.

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_V1 = 'rajvirsingh12/quest-model-merged'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_V1, quantization_config=bnb_config,
    device_map='auto', torch_dtype=torch.float16,
    trust_remote_code=True, attn_implementation='eager',
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_V1, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
print('✅ V1 model loaded')

In [ ]:
# CELL 4: LoRA for DPO
# Same LoRA setup as Phase 2 — DPO adapts another set of LoRA layers on top.

from peft import LoraConfig, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,                       # smaller r for DPO — less drift from v1
    lora_alpha=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

In [ ]:
# CELL 5: DPO Training
#
# DPO loss = log σ(β * (log π(chosen|x)/π_ref(chosen|x) − log π(rejected|x)/π_ref(rejected|x)))
#
# In plain terms: model learns to assign HIGHER probability to chosen quests
# and LOWER probability to rejected quests, relative to the reference model (v1).
#
# β (beta): how much the new model is allowed to diverge from v1
#   high β = stays close to v1 (safer, less learning)
#   low β  = drifts more (faster learning, risk of breaking)
#   0.1 is standard starting value

from trl import DPOTrainer
from transformers import TrainingArguments

OUTPUT_DIR = '/content/drive/MyDrive/ascend_ml/dpo_v2'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,                  # DPO needs fewer epochs than SFT
    per_device_train_batch_size=2,       # DPO uses more VRAM (loads 2 models)
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=5e-6,                  # MUCH lower than SFT (1e-4)
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    evaluation_strategy='steps',
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_steps=25,
    fp16=True,
    optim='paged_adamw_32bit',
    report_to='none',
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,                     # None = uses base model copy as reference
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    peft_config=lora_config,
    beta=0.1,                           # standard DPO β
    max_length=1024,
    max_prompt_length=512,
)

print('🚀 DPO training started...')
dpo_trainer.train()
print('✅ Done')

In [ ]:
# CELL 6: Merge LoRA → push v2 to Hub
ADAPTER_PATH = OUTPUT_DIR + '/final_adapter'
dpo_trainer.model.save_pretrained(ADAPTER_PATH)

from peft import PeftModel
from huggingface_hub import login
import torch

base = AutoModelForCausalLM.from_pretrained(
    MODEL_V1, torch_dtype=torch.float16,
    trust_remote_code=True, device_map='cpu'
)
merged = PeftModel.from_pretrained(base, ADAPTER_PATH).merge_and_unload()

HF_TOKEN    = 'hf_YOUR_WRITE_TOKEN'
HF_USERNAME = 'rajvirsingh12'
REPO_V2     = f'{HF_USERNAME}/quest-model-merged-v2'

login(token=HF_TOKEN)
merged.push_to_hub(REPO_V2)
tokenizer.push_to_hub(REPO_V2)
print(f'✅ V2 pushed → huggingface.co/{REPO_V2}')
print('\nNext: convert to GGUF, upload .gguf to v2 repo, update Space env var')